## Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql import DataFrame
from pyspark.sql.types import *
import re
import pandas as pd

from pathlib import Path

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 00:42:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load

In [2]:
data_path = Path.cwd().parent / 'data' / 'raw'

profile_schema = StructType([
    StructField("age", LongType(), True),
    StructField("credit_card_limit", DoubleType(), True),
    StructField("gender", StringType(), True),
    StructField("id", StringType(), True),
    StructField("registered_on", DateType(), True)
])

offers = (
    spark.read.json(str(data_path / 'offers.json'))
    .withColumnRenamed("id", "offer_id")
)
profile = (
    spark.read.json(str(data_path / 'profile.json'), schema=profile_schema, dateFormat="yyyyMMdd")
    .withColumn("age", F.when(F.col("age")!=118, F.col("age")))
    .withColumnRenamed("id", "account_id")
)
transactions = (
    spark.read.json(str(data_path / 'transactions.json'))
    .withColumn("offer_id", F.coalesce(F.col("value.offer id"), F.col("value.offer_id")))
    .withColumn("transaction_amount", F.col("value.amount"))
    .withColumn("offer_reward", F.col("value.reward"))
    .drop("value")
    .withColumn("transaction_id", F.monotonically_increasing_id())
)

In [5]:
# ---------------------------------------------------------------------------
# Timeline helpers
# ---------------------------------------------------------------------------

def _append_offer_duration(df: DataFrame, offers: DataFrame) -> DataFrame:
    offer_durations = offers.select("offer_id", "duration")
    return (
        df
        .join(offer_durations, "offer_id", "inner")
        .withColumn("ended_at", F.col("received_at") + F.col("duration"))
        .drop("duration")
    )


def _build_timeline(left: DataFrame, right: DataFrame) -> DataFrame:
    return left.unionByName(right)


def _cast_id_columns_safely(df: pd.DataFrame, id_cols: list[str]) -> pd.DataFrame:
    for col in id_cols:
        df[col] = df[col].astype(str).replace({"nan": None, "None": None})
    return df


def _cast_numeric_columns_safely(df: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _sort_timeline_with_priority(pdf: pd.DataFrame, priority_map: dict) -> pd.DataFrame:
    pdf["priority"] = pdf["event_type"].map(priority_map)
    return pdf.sort_values(by=["time", "priority"]).reset_index(drop=True)


def _pop_latest_active_match(active_items: list, event_time: float) -> dict | None:
    for i in range(len(active_items) - 1, -1, -1):
        item = active_items[i]
        if item["received_at"] <= event_time <= item["ended_at"]:
            return active_items.pop(i)
    return None


# ---------------------------------------------------------------------------
# Pandas UDF logic: views
# ---------------------------------------------------------------------------

def _match_views_to_receives(pdf: pd.DataFrame) -> pd.DataFrame:
    VIEW_SCHEMA_COLS = [
        "account_id", "offer_id", "transaction_id", "received_at",
        "ended_at", "view_transaction_id", "viewed_at"
    ]
    ID_COLS = ["transaction_id", "view_transaction_id"]
    NUMERIC_COLS = ["received_at", "ended_at", "viewed_at"]

    pdf = _sort_timeline_with_priority(pdf, {"received": 0, "viewed": 1})

    active_receives = []
    matched_rows = []

    for _, row in pdf.iterrows():
        if row["event_type"] == "received":
            active_receives.append({
                "tx_id": row["tx_id"],
                "received_at": row["time"],
                "ended_at": row["ended_at"],
            })

        elif row["event_type"] == "viewed":
            matched = _pop_latest_active_match(active_receives, row["time"])
            if matched:
                matched_rows.append({
                    "account_id": row["account_id"],
                    "offer_id": row["offer_id"],
                    "transaction_id": matched["tx_id"],
                    "received_at": matched["received_at"],
                    "ended_at": matched["ended_at"],
                    "view_transaction_id": row["tx_id"],
                    "viewed_at": row["time"],
                })

    for unviewed in active_receives:
        matched_rows.append({
            "account_id": pdf["account_id"].iloc[0],
            "offer_id": pdf["offer_id"].iloc[0],
            "transaction_id": unviewed["tx_id"],
            "received_at": unviewed["received_at"],
            "ended_at": unviewed["ended_at"],
            "view_transaction_id": None,
            "viewed_at": None,
        })

    if not matched_rows:
        return pd.DataFrame(columns=VIEW_SCHEMA_COLS)

    out = pd.DataFrame(matched_rows)
    out = _cast_id_columns_safely(out, ID_COLS)
    out = _cast_numeric_columns_safely(out, NUMERIC_COLS)
    return out


# ---------------------------------------------------------------------------
# Pandas UDF logic: completions
# ---------------------------------------------------------------------------

def _match_completions_to_history(pdf: pd.DataFrame) -> pd.DataFrame:
    COMP_SCHEMA_COLS = [
        "account_id", "offer_id", "transaction_id", "received_at", "ended_at",
        "view_transaction_id", "viewed_at", "comp_transaction_id", "completed_at",
    ]
    ID_COLS = ["transaction_id", "view_transaction_id", "comp_transaction_id"]
    NUMERIC_COLS = ["received_at", "ended_at", "viewed_at", "completed_at"]

    pdf = _sort_timeline_with_priority(pdf, {"history": 0, "completed": 1})

    active_histories = []
    matched_rows = []

    for _, row in pdf.iterrows():
        if row["event_type"] == "history":
            active_histories.append({
                "tx_id": row["tx_id"],
                "received_at": row["time"],
                "ended_at": row["ended_at"],
                "view_transaction_id": row["view_transaction_id"],
                "viewed_at": row["viewed_at"],
            })

        elif row["event_type"] == "completed":
            matched = _pop_latest_active_match(active_histories, row["time"])
            if matched:
                matched_rows.append({
                    "account_id": row["account_id"],
                    "offer_id": row["offer_id"],
                    "transaction_id": matched["tx_id"],
                    "received_at": matched["received_at"],
                    "ended_at": matched["ended_at"],
                    "view_transaction_id": matched["view_transaction_id"],
                    "viewed_at": matched["viewed_at"],
                    "comp_transaction_id": row["tx_id"],
                    "completed_at": row["time"],
                })

    for uncompleted in active_histories:
        matched_rows.append({
            "account_id": pdf["account_id"].iloc[0],
            "offer_id": pdf["offer_id"].iloc[0],
            "transaction_id": uncompleted["tx_id"],
            "received_at": uncompleted["received_at"],
            "ended_at": uncompleted["ended_at"],
            "view_transaction_id": uncompleted["view_transaction_id"],
            "viewed_at": uncompleted["viewed_at"],
            "comp_transaction_id": None,
            "completed_at": None,
        })

    if not matched_rows:
        return pd.DataFrame(columns=COMP_SCHEMA_COLS)

    out = pd.DataFrame(matched_rows)
    out = _cast_id_columns_safely(out, ID_COLS)
    out = _cast_numeric_columns_safely(out, NUMERIC_COLS)
    return out


# ---------------------------------------------------------------------------
# Public pipeline stages
# ---------------------------------------------------------------------------

def get_received_events(transactions: DataFrame) -> DataFrame:
    return (
        transactions
        .where(F.col("event") == "offer received")
        .select(
            F.col("transaction_id"),
            F.col("account_id"),
            F.col("offer_id"),
            F.col("time_since_test_start").alias("received_at"),
        )
    )


def get_offer_end(received: DataFrame, offers: DataFrame) -> DataFrame:
    return _append_offer_duration(received, offers)


def get_views(received_with_end: DataFrame, transactions: DataFrame) -> DataFrame:
    raw_views = (
        transactions
        .where(F.col("event") == "offer viewed")
        .select(
            F.col("account_id"),
            F.col("offer_id"),
            F.col("transaction_id").alias("tx_id"),
            F.col("time_since_test_start").alias("time"),
            F.lit("viewed").alias("event_type"),
            F.lit(None).cast("double").alias("ended_at"),
        )
    )

    receives_as_timeline_events = received_with_end.select(
        "account_id", "offer_id",
        F.col("transaction_id").alias("tx_id"),
        F.col("received_at").alias("time"),
        F.lit("received").alias("event_type"),
        F.col("ended_at"),
    )

    view_output_schema = """
        account_id string, offer_id string,
        transaction_id string, received_at double, ended_at double,
        view_transaction_id string, viewed_at double
    """

    return (
        _build_timeline(receives_as_timeline_events, raw_views)
        .groupBy("account_id", "offer_id")
        .applyInPandas(_match_views_to_receives, schema=view_output_schema)
    )


def get_complete(views: DataFrame, transactions: DataFrame) -> DataFrame:
    raw_completions = (
        transactions
        .where(F.col("event") == "offer completed")
        .select(
            F.col("account_id"),
            F.col("offer_id"),
            F.col("transaction_id").alias("tx_id"),
            F.col("time_since_test_start").alias("time"),
            F.lit("completed").alias("event_type"),
            F.lit(None).cast("double").alias("ended_at"),
            F.lit(None).cast("string").alias("view_transaction_id"),
            F.lit(None).cast("double").alias("viewed_at"),
        )
    )

    views_as_timeline_history = views.select(
        "account_id", "offer_id",
        F.col("transaction_id").alias("tx_id"),
        F.col("received_at").alias("time"),
        F.lit("history").alias("event_type"),
        F.col("ended_at"),
        F.col("view_transaction_id"),
        F.col("viewed_at"),
    )

    completion_output_schema = """
        account_id string, offer_id string,
        transaction_id string, received_at double, ended_at double,
        view_transaction_id string, viewed_at double,
        comp_transaction_id string, completed_at double
    """

    return (
        _build_timeline(views_as_timeline_history, raw_completions)
        .groupBy("account_id", "offer_id")
        .applyInPandas(_match_completions_to_history, schema=completion_output_schema)
    )


# ---------------------------------------------------------------------------
# Pipeline execution
# ---------------------------------------------------------------------------

def build_offer_lifecycle(transactions: DataFrame, offers: DataFrame) -> DataFrame:
    return (
        get_received_events(transactions)
        .transform(get_offer_end, offers)
        .transform(get_views, transactions)
        .transform(get_complete, transactions)
    )


def filter_to_single_offer(df: DataFrame, account_id: str, offer_id: str) -> DataFrame:
    return df.where(
        (F.col("account_id") == account_id) &
        (F.col("offer_id") == offer_id)
    )


build_offer_lifecycle(transactions, offers).write.mode("overwrite").parquet("../data/processed/offer_lifecycle")

## How many offers does each customer receive?

In [0]:
display(
    transactions
    .where(F.col("event")=="offer received")
    .groupBy("account_id")
    .agg(F.count("*").alias("n_offer"))
    .groupBy("n_offer")
    .count()
    .orderBy("n_offer")
)

Databricks visualization. Run in Databricks to view.

## What are the possible sequences of events?

In [0]:
display(
    transactions
    .where(F.col("event").isin("offer viewed"))
    .withColumn("transaction_id", F.col("transaction_id").cast("string"))
    .join(df.select("view_transaction_id"), transactions.transaction_id == df.view_transaction_id, "left_anti")
)

In [0]:
display(
    transactions
    .where(F.col("event")!="transaction")
    .where(F.col("account_id") == "270e7fd65f7e45c58b79d0d8ad2c72ab")
    .orderBy("time_since_test_start")
)

In [0]:
df = (
    get_received_events(transactions)
    .transform(get_offer_end, offers)
    .transform(get_views, transactions)
    
    .where(
        (F.col("account_id") == "e7543efb4ab449c48de222ed0b689db9") &
        (F.col("offer_id") == "fafdcd668e3743c1bb461111dcafc2a4") 
    )
)

display(df)

In [0]:
display(
    transactions
    .where(
        (F.col("offer_id")=="fafdcd668e3743c1bb461111dcafc2a4") &
        (F.col("account_id") == "e7543efb4ab449c48de222ed0b689db9")
        # (F.col("event") == "offer viewed")
    )
)

In [0]:
import pyspark.sql.functions as F
import pandas as pd

# ==========================================
# 1. Preparação: Unificar tudo na mesma linha do tempo
# ==========================================
df_receives = (
    transactions
    .where(F.col("event") == "offer received")
    .join(offers, "offer_id", "inner")
    .select(
        F.col("account_id"),
        F.col("offer_id"),
        F.lit("received").alias("event_type"),
        F.col("time_since_test_start").alias("time"),
        F.col("transaction_id").alias("tx_id"),
        F.col("duration").cast("double") # Lembre-se do * 24 se for dias!
    )
)

df_views = (
    transactions
    .where(F.col("event") == "offer viewed")
    .select(
        F.col("account_id"),
        F.col("offer_id"),
        F.lit("viewed").alias("event_type"),
        F.col("time_since_test_start").alias("time"),
        F.col("transaction_id").alias("tx_id"),
        F.lit(0.0).alias("duration") 
    )
)

df_timeline = df_receives.unionByName(df_views)

# ==========================================
# 2. Definindo o Schema de Saída (Obrigatório no Pandas UDF)
# ==========================================
output_schema = """
    account_id string, 
    offer_id string, 
    transaction_id string, 
    view_transaction_id string, 
    received_at double, 
    ended_at double, 
    viewed_at double
"""

# ==========================================
# 3. A Lógica de Iteração FIFO (Pandas UDF)
# ==========================================
def process_fifo_pandas(pdf: pd.DataFrame) -> pd.DataFrame:
    # 1. Ordena os eventos cronologicamente dentro do Pandas
    pdf['priority'] = pdf['event_type'].map({'received': 0, 'viewed': 1})
    pdf = pdf.sort_values(by=['time', 'priority'])
    
    active_receives = []
    matched_results = []
    
    # 2. Itera nas linhas desse cliente/oferta
    for _, row in pdf.iterrows():
        if row['event_type'] == 'received':
            active_receives.append({
                "receive_tx_id": row['tx_id'],
                "received_at": row['time'],
                "ended_at": row['time'] + row['duration']
            })
            
        elif row['event_type'] == 'viewed':
            view_time = row['time']
            view_tx_id = row['tx_id']
            
            matched_idx = -1
            for i, rec in enumerate(active_receives):
                if view_time >= rec["received_at"] and view_time <= rec["ended_at"]:
                    matched_idx = i
                    break 
            
            if matched_idx != -1:
                rec = active_receives.pop(matched_idx)
                
                matched_results.append({
                    "account_id": row['account_id'],
                    "offer_id": row['offer_id'],
                    "transaction_id": rec["receive_tx_id"],
                    "view_transaction_id": view_tx_id,
                    "received_at": rec["received_at"],
                    "ended_at": rec["ended_at"],
                    "viewed_at": view_time
                })
                
    # 3. Retorna DataFrame garantindo a tipagem idêntica ao DDL Schema
    if not matched_results:
        return pd.DataFrame(columns=[
            "account_id", "offer_id", "transaction_id", "view_transaction_id", 
            "received_at", "ended_at", "viewed_at"
        ])
        
    df_out = pd.DataFrame(matched_results)
    
    # --- CORREÇÃO AQUI ---
    # Forçamos o cast para string e float para o PyArrow não reclamar
    df_out['transaction_id'] = df_out['transaction_id'].astype(str)
    df_out['view_transaction_id'] = df_out['view_transaction_id'].astype(str)
    df_out['received_at'] = df_out['received_at'].astype(float)
    df_out['ended_at'] = df_out['ended_at'].astype(float)
    df_out['viewed_at'] = df_out['viewed_at'].astype(float)
    
    return df_out

# ==========================================
# 4. Aplicação do mapInPandas via groupByKey
# ==========================================
final_df = (
    df_timeline
    .groupBy("account_id", "offer_id") # Agrupa garantindo o isolamento
    .applyInPandas(process_fifo_pandas, schema=output_schema)
)

display(
    final_df
    .where(
        (F.col("account_id") == "e7543efb4ab449c48de222ed0b689db9") &
        (F.col("offer_id") == "fafdcd668e3743c1bb461111dcafc2a4") 
    )
)

## How many customers receive multiple offers simultaneously?

In [0]:
offer_end_rows = (
    transactions
    .where(F.col("event") == "offer received")
    .join(offers.select("offer_id", "duration"), "offer_id", "left")
    .withColumn("event", F.lit("offer end"))
    .withColumn("time_since_test_start", F.col("time_since_test_start") + F.col("duration"))
    .drop("duration")
)

transactions_with_offer_end = transactions.unionByName(offer_end_rows)

days = spark.createDataFrame([(i,) for i in range(30)], ["time_since_test_start"])
all_days = profile.select("account_id").crossJoin(days)

w = Window.partitionBy("account_id").orderBy("time_since_test_start")

display(
    transactions_with_offer_end
    .withColumn("time_since_test_start", F.floor(F.col("time_since_test_start")))
    .join(all_days, ["account_id", "time_since_test_start"], "full")

    .groupBy("account_id", "time_since_test_start")
    .agg(
        F.sum(F.when(F.col("event")=="offer received", 1).otherwise(0)).alias("offer_received"),
        F.sum(F.when(F.col("event").isin("offer end", "offer_reward"), 1).otherwise(0)).alias("offer_end")
    )
    .withColumn("n_offers", F.sum("offer_received").over(w) - F.sum("offer_end").over(w))

    .groupBy("account_id")
    .agg(F.max("n_offers").alias("max_n_offers"))

    .groupBy("max_n_offers")
    .count()
)

Databricks visualization. Run in Databricks to view.

## How many receive offers with different offer_ids?

In [0]:
# offers.select("offer_id", "duration")

display(
    transactions
    .where(F.col("event")!="transaction")

    .where(F.col("account_id") == "004c5799adbf42868b9cff0396190900")
)


In [0]:
offer_end_rows = (
    transactions
    .where(F.col("event") == "offer received")
    .join(offers.select("offer_id", "duration"), "offer_id", "left")
    .withColumn("event", F.lit("offer end"))
    .withColumn("time_since_test_start", F.col("time_since_test_start") + F.col("duration"))
    .drop("duration")
)

transactions_with_offer_end = transactions.unionByName(offer_end_rows)

days = spark.createDataFrame([(i,) for i in range(30)], ["time_since_test_start"])
all_days = profile.select("account_id").crossJoin(days)

w = Window.partitionBy("account_id", "offer_id").orderBy("time_since_test_start", "event_order")

display(
    transactions_with_offer_end
    .withColumn("time_since_test_start", F.floor(F.col("time_since_test_start")))
    .join(all_days, ["account_id", "time_since_test_start"], "full")
    .withColumn("active_flag", F.when(F.col("event") == "offer received", 1).when(F.col("event") == "offer end", -1).otherwise(0))
    .withColumn("event_order", F.when(F.col("event") == "offer received", 1).when(F.col("event") == "offer end", 2))
    .withColumn("concurrent_active_offers", F.sum("active_flag").over(w))

    .where(F.col("account_id") == "004c5799adbf42868b9cff0396190900")
)

## How many accounts receive any single offer?

In [0]:
offer_end_rows = (
    transactions
    .where(F.col("event") == "offer received")
    .join(offers.select("offer_id", "duration"), "offer_id", "left")
    .withColumn("event", F.lit("offer end"))
    .withColumn("time_since_test_start", F.col("time_since_test_start") + F.col("duration"))
    .drop("duration")
)

transactions_with_offer_end = transactions.unionByName(offer_end_rows)

days = spark.createDataFrame([(i,) for i in range(30)], ["time_since_test_start"])
all_days = profile.select("account_id").crossJoin(days)

w = Window.partitionBy("account_id").orderBy("time_since_test_start")

display(
    transactions_with_offer_end

    .withColumn("time_since_test_start", F.floor(F.col("time_since_test_start")))
    .join(all_days, ["account_id", "time_since_test_start"], "full")

    .groupBy("account_id", "time_since_test_start")
    .agg(
        F.sum(F.when(F.col("event")=="offer received", 1).otherwise(0)).alias("offer_received"),
        F.sum(F.when(F.col("event").isin("offer end", "offer_reward"), 1).otherwise(0)).alias("offer_end")
    )
    .withColumn("n_offers", F.sum("offer_received").over(w) - F.sum("offer_end").over(w))

    # .where(F.col("account_id") == "c5809b646a4b48ec82424067ce2e639e")
    .groupBy("account_id")
    .agg(
        F.concat_ws("", F.collect_list(F.col("n_offers").cast("string"))).alias("seq")
    )
    .withColumn("any_single", F.col("seq").rlike("(^|0)1+(0|$)"))
    .groupBy("any_single")
    .count()
)